# 1 — Create `df_log` from the log files

**This notebook makes the dataset. It does not analyse it** — that is notebook 2.

Everything here comes from `log.json` and nothing else: no video, no optic flow. That is what lets
it run over a whole server directory whose sessions were never put through the video pipeline.

It produces exactly **two tables**:

| | one row per | holds |
|---|---|---|
| `df_sessions` | SESSION | who/when/which world/which protocol + **the performance numbers (D, chance, throughput…)** |
| `df_trials` | TRIAL | the trial window, path geometry, the on-screen icons, the outcome, the cluster and the error/conflict labels |

Both are saved to `MAIN_DIR/df_log/`. Notebook 2 loads them and never touches the server again.

**The trial logic is not written here.** `task_*/build_trials.py` already defines a trial correctly —
a **spawn batch**, so a batch that ends without a collection (a reshuffle, the session-end tail) is a
real row — and it carries the `icons` list, the `NORMAL`/`BANISH_WORLD` column, and
`start_frame`/`end_frame`. Those builders are called here directly, and **nothing is read from or
written into the session folders**.

Sections **A** are sanity checks on the data; sections **B** build; section **C** verifies and saves.

## Config ← YOU SET THIS

In [ ]:
MAIN_DIR = '/mnt/server/data'     # the directory holding one folder per animal
VIEW_SCALES = {}                  # {world signature: scale} for any world without a known scale
PIPELINE_DIR = None               # None = locate session_pipeline/ automatically
# =============================================================================

import sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
if PIPE is None:
    raise FileNotFoundError('set PIPELINE_DIR. Tried: ' + ', '.join(str(c) for c in cands))
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'performance'))

import session_index as sidx, perf_from_log as pfl, build_log_df as bl
# RELOAD, don't just import: Python caches a module, so after a `git pull` the kernel would keep
# running the OLD code while re-running this cell looked like it worked.
sidx = importlib.reload(sidx); pfl = importlib.reload(pfl); bl = importlib.reload(bl)

print(f'pipeline : {PIPE}')
print(f'main dir : {MAIN_DIR}')
print(f'builders : {list(bl.TASK_MODULES)}   (others fall back to the generic builder)')

## A1 — SANITY: what is actually on disk?

Deliberately dumb: it lists folders and counts files, **without opening a single log**. If the mount
is missing, the path is mis-typed, or a sync is half-finished, it looks wrong *here* — rather than
showing up later as a mysteriously small dataset.

In [ ]:
ANIMALS = sidx.list_animals(MAIN_DIR)
display(ANIMALS)

for a in ANIMALS.animal:
    print(f'\n--- {a} ---')
    display(sidx.list_sessions(MAIN_DIR / a).head(8))

## A2 — What a log actually contains

One log, opened and shown, so the rest of the notebook is readable: you can see the fields every
column below is derived from.

In [ ]:
_first = next((MAIN_DIR / a).glob('*/log.json'), None) or next(MAIN_DIR.glob('*/*/log.json'))
_L = json.load(open(_first))
print(_first, '\n')
print('top-level keys :', list(_L.keys()), '\n')
print('experiment_data:', _L.get('experiment_data'), '\n')
print('collected[0]   :', (_L.get('collected') or [{}])[0], '\n')
print('spawns[0]      :', {k: v for k, v in (_L.get('spawns') or [{}])[0].items() if k != 'current'})
print('worlds[0]      :', (_L.get('worlds') or [{}])[0], '\n')
print('n collected    :', len(_L.get('collected') or []))
print('protocol       :', pfl.classify_task(pfl.log_effects(_L),
                                            has_multiplier=pfl.log_has_multiplier(_L)))

## A3 — SANITY: read every log and check the identities

`discover()` opens each log and reads three things it refuses to guess: the **date**
(`experiment_data.datetime`), the **animal** (the ID's digits, falling back to the folder name), and
the **protocol** (from the icons the board OFFERED, via the spawn stream — never from what the animal
happened to collect, which would drop exactly the sessions where he avoided the hazard).

Everything printed here is a **check**, not a result.

In [ ]:
S = pd.concat([sidx.discover(MAIN_DIR / a, view_scales=VIEW_SCALES) for a in ANIMALS.animal],
               ignore_index=True)
print(f'\n{len(S)} session(s), {S.mouse.nunique()} animal(s): {sorted(S.mouse.dropna().unique())}')

# animal resolved from the FOLDER rather than the log ID?
if (S.mouse_src == 'folder').any():
    n = int((S.mouse_src == 'folder').sum())
    print(f'\n{n} session(s) took the animal from the folder name (the log ID had no number):')
    display(S.loc[S.mouse_src == 'folder', ['session', 'mouse_raw', 'mouse']].head())

# days holding more than one session
dup = S.day.duplicated(keep=False)
if dup.any():
    print('\ndays with more than one session:')
    display(S.loc[dup, ['session', 'day', 'time', 'name']])

display(S[['session', 'mouse', 'day', 'task', 'world', 'view_scale', 'use', 'note']].head(12))

### A4 — SANITY: which worlds, which protocols, and do they agree?

`protocol_census` numbers the worlds `W1…Wn` and cross-tabs them against protocol.
`world_protocol_audit` then checks that against the rule that a world implies a protocol
(**W4 = banishment, W3 = timeout**) and, for any disagreement, prints the evidence — the icons the
log **offered** versus what was **collected**. A session on a banishment world whose log never
spawned a `banish` icon is a fact about the log, not a classifier bug, and the printout says which
of the two you are looking at.

In [ ]:
CENSUS = sidx.protocol_census(S)      # also assigns S['world_id']
display(CENSUS)

AUDIT = sidx.world_protocol_audit(S)

## B1 — BUILD the two tables

One call. For each session: the task's `build_trials` → `cluster_paths` → `label_trials` (all
log-only), then the performance numbers merged into the session row.

**Every discovered session becomes a row.** One that cannot be scored keeps its identity, gets `NaN`
performance and a reason in `perf_error`, and stays in the table — filtering is notebook 2's
explicit choice, not a deletion baked in here. Days holding two sessions are **flagged**
(`is_dup_day`, `keep_of_day`), not dropped.

In [ ]:
df_sessions, df_trials = bl.build_all(MAIN_DIR, view_scales=VIEW_SCALES)

## C1 — VERIFY the dataset

Checks that it is right, not that it is interesting. Anything printing `FAIL` needs looking at
before the tables are used.

In [ ]:
def check(name, ok, detail=''):
    print(f'  [{"PASS" if ok else "FAIL"}] {name}' + (f'   {detail}' if detail else ''))

print('DATASET CHECKS')
check('every session has a unique name', df_sessions.session.is_unique)
check('every trial belongs to a listed session',
      set(df_trials.session) <= set(df_sessions.session))

# trial counts agree between the two tables
per = df_trials.groupby('session').size()
agree = all(int(per.get(r.session, 0)) == int(r.n_trials_total) for _, r in df_sessions.iterrows())
check('trials per session match df_sessions.n_trials_total', agree)

# reward drops computed two independent ways
d_tr = df_trials.groupby('session').drops.sum()
d_se = df_sessions.set_index('session').drops
both = [s for s in d_se.index if s in d_tr.index and pd.notna(d_se[s])]
check('reward drops agree (trial sum vs session total)',
      all(abs(float(d_tr[s]) - float(d_se[s])) < 1e-6 for s in both),
      f'{len(both)} session(s)')

check('no session is missing a world', df_sessions.world_sig.notna().all())
n_bad = int((df_sessions.perf_error != '').sum())
check('all sessions scored', n_bad == 0, f'{n_bad} without performance')
if n_bad:
    display(df_sessions.loc[df_sessions.perf_error != '', ['session', 'task', 'perf_error']])

gen = df_trials.builder.eq('generic').sum() if 'builder' in df_trials else 0
if gen:
    print(f'\n  NOTE {gen} trial(s) came from the GENERIC builder (no dedicated one for that '
          f'protocol yet): {sorted(df_trials.loc[df_trials.builder == "generic", "task"].unique())}')
    print('       They have the common columns only -- no spawn-batch trials, no cluster/labels.')

### C2 — Cross-check against a session built by the full pipeline

Where a session has already been through the video pipeline it has its own `df_trials_clean.pkl`,
built by the same task builder. This asserts the log-only table reproduces it — the strongest
available check that nothing was lost by building from the raw folder.

In [ ]:
for _, r in df_sessions.iterrows():
    ref_p = Path(r['dir']) / 'df_trials_clean.pkl'
    if not ref_p.exists():
        continue
    mine = df_trials[df_trials.session == r.session]
    # Only compare where BOTH tables were built by the same task builder. A generic-builder session
    # defines a trial as collection-to-collection while the reference may be a spawn-batch table
    # (or, for the legacy JPAS_0231 file, an older overlapping-window schema entirely) -- comparing
    # those reports a difference of DEFINITION as though it were an error, and a check that cries
    # wolf is worse than no check.
    if 'builder' in mine and (mine.builder == 'generic').any():
        print(f'{r.session}:  SKIPPED -- built by the generic builder, so the reference table is '
              f'a different trial definition, not a comparable one')
        continue
    ref = pd.read_pickle(ref_p)
    print(f'{r.session}:  mine {len(mine)} trials vs reference {len(ref)}')
    if len(mine) != len(ref):
        check('   same number of trials', False)
        continue
    check('   outcomes identical', (mine.outcome.values == ref.outcome.values).all())
    cols = [c for c in ['dur_s', 'path_efficiency', 'time_in_corner', 'mean_speed',
                        'heading_align'] if c in mine and c in ref]
    check('   geometry identical',
          all(np.allclose(mine[c].values.astype(float), ref[c].values.astype(float),
                          equal_nan=True) for c in cols), f'({", ".join(cols)})')
    if 'cluster_name' in ref and 'cluster_name' in mine:
        check('   cluster identical', (mine.cluster_name.values == ref.cluster_name.values).all())

## C3 — SAVE

`.pkl` keeps everything, including the per-trial coordinate arrays and the `icons` lists. The
`.csv` of `df_sessions` is for eyeballing and sharing (it cannot carry the build stamp).

Each file is stamped with when and by what version it was built, so a stale copy is recognisable
rather than merely old.

In [ ]:
OUT = MAIN_DIR / 'df_log'
OUT.mkdir(exist_ok=True)

bl.save(df_sessions, OUT / 'df_sessions.pkl')
bl.save(df_trials,   OUT / 'df_trials.pkl')
bl.save(df_sessions, OUT / 'df_sessions.csv')

print('\nnotebook 2 reads these and needs nothing else.')
df_sessions.head()